In [26]:
import arcpy
import psycopg
import csv
from dotenv import load_dotenv
import os

In [27]:
def execute_and_append_to_csv(conn, query, output_file, title=None):
    cursor = conn.cursor()

    with conn.transaction():
        cursor.execute(query)

        # Fetch all rows and column names
        rows = cursor.fetchall()
        colnames = [desc[0] for desc in cursor.description]

        # Open the CSV file in append mode
        with open(output_file, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            # Write the title, if provided
            if title:
                writer.writerow([title])  # Write the title row
                writer.writerow([])  # Add a blank line for spacing

            # Add header (column names) row before the data
            writer.writerow(colnames)

            # Write data rows
            writer.writerows(rows)

            # Add a blank line after the data for separation
            writer.writerow([])



        print(f"Data successfully appended with section: {title}")


In [28]:
load_dotenv()

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
sde_path = os.getenv('SDE_PATH')
arcpy.env.workspace = sde_path
schema = os.getenv('SCHEMA')
db = os.getenv('DB_NAME')
csv_out_path = os.getenv('CSV_OUTPUT_PATH')
arcpy.env.overwriteOutput = True

In [29]:
conn = psycopg.connect(
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    autocommit=True
)


In [ ]:
# Subset creation for treatments of interest

ti_stats_table = 'treatment_index_fy_21_25_stats'
base_treatment_index = 'treatment_index'

cursor = conn.cursor()

with conn.transaction():
    cursor.execute(f'''
        DROP TABLE IF EXISTS
        {schema}.{ti_stats_table};
    ''')

    cursor.execute(f'''
        CREATE TABLE 
            {schema}.{ti_stats_table}
        AS
        SELECT * FROM
            {schema}.{base_treatment_index}
        WHERE 
            treatment_date BETWEEN '2020-10-01 00:00:00' AND '2025-09-30 23:59:59'
            AND status = 'Completed'
            AND identifier_database in ('IFPRS', 'NFPORS', 'FACTS Hazardous Fuels', 'FACTS Common Attributes')
            AND (error IS NULL
   			OR (error NOT LIKE '%DUPLICATE_DROP%' AND error NOT LIKE '%HIGH_COST%'));
    ''') 

arcpy.management.AddSpatialIndex(
    in_features=rf"{sde_path}\{db}.{schema}.{ti_stats_table}",
    spatial_grid_1=0,
    spatial_grid_2=0,
    spatial_grid_3=0
)
# arcpy.management.RegisterWithGeodatabase(
# in_dataset=rf"{sde_path}\{db}.{schema}.{ti_stats_table}",
# in_object_id_field="objectid",
# in_shape_field="shape",
# in_geometry_type="POLYGON",
# in_spatial_reference='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 1000;-100000 1000;8.98315284119521E-09;0.001;0.001;IsHighPrecision',
# in_extent=None
# )

In [ ]:
# Recent Wildifires Import

# Recent Wildfires URL
recent_fires_service_url = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters/FeatureServer/0"
recent_fires_fl_name = "recent_fires_fy_21_25"

recent_fires_query = "poly_PolygonDateTime BETWEEN date '2020-10-01' AND date '2025-09-30'"

arcpy.management.MakeFeatureLayer(recent_fires_service_url, recent_fires_fl_name, where_clause=recent_fires_query)

# import recent wildfires into postgres
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=recent_fires_fl_name,
    Output_Geodatabase=sde_path
)

In [ ]:
# Federal Lands

federal_lands_service_url = "https://services5.arcgis.com/7weheFjxuNkGGiZi/ArcGIS/rest/services/USA_Federal_Lands_2025/FeatureServer/0"
federal_lands_fl_name = "us_federal_lands"

arcpy.management.MakeFeatureLayer(federal_lands_service_url, federal_lands_fl_name)

federal_lands_wgs84 = 'us_federal_lands_wgs84'

arcpy.management.Project(
    in_dataset="us_federal_lands",
    out_dataset=rf"{sde_path}\{db}.{schema}.{federal_lands_wgs84}",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    transform_method=None,
    in_coor_system='PROJCRS["WGS_1984_Web_Mercator_Auxiliary_Sphere",BASEGEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Mercator_Auxiliary_Sphere",METHOD["Mercator_Auxiliary_Sphere"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",0.0],PARAMETER["Standard_Parallel_1",0.0],PARAMETER["Auxiliary_Sphere_Type",0.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["EPSG","3857"]]',
    preserve_shape="NO_PRESERVE_SHAPE",
    max_deviation=None,
    vertical="NO_VERTICAL"
)



In [ ]:
# Creation of 'BIL' Treatments Layer to Intersect with Federal Lands

ti_stats_bil_table = 'treatment_index_fy_21_25_stats_bil'


cursor = conn.cursor()

with conn.transaction():
    cursor.execute(f'''
        DROP TABLE IF EXISTS
        {schema}.{ti_stats_bil_table};
    ''')

    cursor.execute(f'''
        CREATE TABLE 
            {schema}.{ti_stats_bil_table}
        AS
        SELECT * FROM
            {schema}.{ti_stats_table}
        WHERE 
            fund_source = 'BIL'
    ''')

arcpy.management.AddSpatialIndex(
    in_features=rf"{sde_path}\{db}.{schema}.{ti_stats_bil_table}",
    spatial_grid_1=0,
    spatial_grid_2=0,
    spatial_grid_3=0
)   
# arcpy.management.RegisterWithGeodatabase(
# in_dataset=rf"{sde_path}\{db}.{schema}.{ti_stats_bil_table}",
# in_object_id_field="objectid",
# in_shape_field="shape",
# in_geometry_type="POLYGON",
# in_spatial_reference='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 1000;-100000 1000;8.98315284119521E-09;0.001;0.001;IsHighPrecision',
# in_extent=None
# )

In [ ]:
ti_bil_agency_dissolve_layer = 'treatment_index_stats_agency_dissolve'

arcpy.analysis.PairwiseDissolve(
    in_features=rf"{sde_path}\{db}.{schema}.{ti_stats_bil_table}",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{ti_bil_agency_dissolve_layer}",
    dissolve_field="agency",
    statistics_fields=None,
    multi_part="MULTI_PART",
    concatenation_separator="",
    out_lineage_table=None
)

In [ ]:
fed_lands_treatments_intersect_layer = 'treatment_index_fy_21_25_stats_bil_intersect_fed_lands'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{federal_lands_wgs84}';'{sde_path}\{db}.{schema}.{ti_stats_bil_table}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{fed_lands_treatments_intersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

In [ ]:
fed_lands_treatment_intersect_agency_dissolve_layer = 'treatment_index_fy_21_25_stats_bil_intersect_fed_lands_dissolve'

arcpy.analysis.PairwiseDissolve(
    in_features=rf"{sde_path}\{db}.{schema}.{fed_lands_treatments_intersect_layer}",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{fed_lands_treatment_intersect_agency_dissolve_layer}",
    dissolve_field="agency",
    statistics_fields=None,
    multi_part="MULTI_PART",
    concatenation_separator="",
    out_lineage_table=None
)

In [ ]:
output_file = csv_out_path

title = '% of Total Treatments that Intersect a Wildfire (Treatment Before Wildfire)'

query = f'''

    SELECT
        (COUNT(DISTINCT ti.unique_id) * 100.0 /
        (SELECT COUNT(*) FROM {schema}.{ti_stats_table})
    ) AS percent_intersect
    
    FROM {schema}.{ti_stats_table} ti
        WHERE EXISTS (
            SELECT 1
            FROM {recent_fires_fl_name} recent_fires
            WHERE ST_Intersects(recent_fires.shape, ti.shape)
            AND ti.treatment_date < recent_fires.poly_polygondatetime
        );

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = '% of Wildfires that Intersect a Treatment (Treatment Before Wildfire)'

query = f'''

    SELECT
        (COUNT(DISTINCT recent_fires.poly_irwinid) * 100.0 /  -- Selection
        (SELECT COUNT(*) FROM {schema}.{recent_fires_fl_name})  -- Full Layer Count
    ) AS percent_intersect
    
    FROM {schema}.{recent_fires_fl_name} recent_fires
        WHERE EXISTS (
            SELECT 1
            FROM {schema}.{ti_stats_table} ti
            WHERE ST_Intersects(recent_fires.shape, ti.shape)
            AND ti.treatment_date < recent_fires.poly_polygondatetime  
        );

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = '% of IIJA Treatments That Intersect a Wildfire (Treatment Before Wildfire)'

query = f'''

    SELECT
        (COUNT(DISTINCT ti.unique_id) * 100.0 /   -- BIL overlap treatments
        (SELECT COUNT(*) FROM {schema}.{ti_stats_table} where fund_source = 'BIL')  --All BIL Treatments
    ) AS percent_intersect
    
    FROM {schema}.{ti_stats_table} ti
        WHERE EXISTS (
            SELECT 1
            FROM {recent_fires_fl_name} recent_fires
            WHERE ST_Intersects(recent_fires.shape, ti.shape)
            AND ti.treatment_date < recent_fires.poly_polygondatetime
            AND fund_source = 'BIL'
        );

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = '% of Wildfires That Intersect an IIJA Treatment (Treatment Before Wildfire)'

query = f'''

    SELECT
        (COUNT(DISTINCT recent_fires.poly_irwinid) * 100.0 /  -- Selection
        (SELECT COUNT(*) FROM {schema}.{recent_fires_fl_name})  -- Full Layer Count
    ) AS percent_intersect
    
    FROM {schema}.{recent_fires_fl_name} recent_fires
        WHERE EXISTS (
            SELECT 1
            FROM {schema}.{ti_stats_table} ti
            WHERE ST_Intersects(recent_fires.shape, ti.shape)
            AND ti.treatment_date < recent_fires.poly_polygondatetime
            AND fund_source = 'BIL'
        );

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'IIJA Total Acres treated by Agency (Federal Lands Intersect)'

query = f'''

    SELECT 
    	agency,         
    	SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM
        {schema}.{fed_lands_treatments_intersect_layer}
    GROUP BY 
        agency
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'IIJA Total Acres Treated by Agency (Federal Lands Intersect Dissolved)'

query = f'''

    SELECT 
    	agency,         
    	ST_Area(ST_Transform(shape, 5070)) / 4046.8564224 AS acres
    FROM
        {schema}.{fed_lands_treatment_intersect_agency_dissolve_layer}
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'IIJA Total Acres Treated by Agency (Treatment Index Agency Field)'

query = f'''

    SELECT 
    	agency,         
    	SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM
        {schema}.{ti_stats_table}
    WHERE 
        fund_source = 'BIL'
    GROUP BY 
        agency
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'IIJA Total Acres Treated by Agency (Treatment Index Agency Field Dissolved)'

query = f'''

SELECT 
    	agency,         
    	ST_Area(ST_Transform(shape, 5070)) / 4046.8564224 AS acres
    FROM
       {schema}.{ti_bil_agency_dissolve_layer}
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'Fed Land Acreage Totals for % Calculation'

query = f'''

    SELECT 
    	agency,         
    	SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM
        {schema}.us_federal_lands_wgs84
    GROUP BY 
        agency
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'Total US Funding by Treatment Category'

query = f'''

    SELECT 
    	twig_category,         
    	SUM(total_cost) AS total_funding
    FROM
        {schema}.{ti_stats_table}
    GROUP BY 
        twig_category
    ORDER BY 
        total_funding desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
title = 'IIJA Funding by Agency'

query = f'''

    SELECT
        agency as agency,
        sum(total_cost) as total_funding
    FROM
        staging.{ti_stats_table}
    WHERE
        fund_source = 'BIL'
    GROUP BY agency
    ORDER BY total_funding desc;
    
'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
# NEW CELL


fed_lands_treatments_intersect_layer_full = 'treatment_index_fy_21_25_stats_intersect_fed_lands'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{federal_lands_wgs84}';'{sde_path}\{db}.{schema}.{ti_stats_table}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{fed_lands_treatments_intersect_layer_full}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

In [ ]:
# NEW CELL

fed_lands_treatment_intersect_agency_dissolve_layer_full = 'treatment_index_fy_21_25_stats_intersect_fed_lands_dissolve'

arcpy.analysis.PairwiseDissolve(
    in_features=rf"{sde_path}\{db}.{schema}.{fed_lands_treatments_intersect_layer_full}",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{fed_lands_treatment_intersect_agency_dissolve_layer_full}",
    dissolve_field="agency",
    statistics_fields=None,
    multi_part="MULTI_PART",
    concatenation_separator="",
    out_lineage_table=None
)

In [ ]:
# NEW CELL

title = 'Total Acres treated by Agency (Federal Lands Intersect)'

query = f'''

    SELECT 
    	agency,         
    	SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM
        {schema}.{fed_lands_treatments_intersect_layer_full}
    GROUP BY 
        agency
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
# NEW CELL

title = 'Total Acres Treated by Agency (Federal Lands Intersect Dissolved)'

query = f'''

    SELECT 
    	agency,         
    	ST_Area(ST_Transform(shape, 5070)) / 4046.8564224 AS acres
    FROM
        {schema}.{fed_lands_treatment_intersect_agency_dissolve_layer_full}
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
# NEW CELL

ti_agency_dissolve_layer = 'treatment_index_stats_agency_dissolve'

arcpy.analysis.PairwiseDissolve(
    in_features=rf"{sde_path}\{db}.{schema}.{ti_stats_table}",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{ti_agency_dissolve_layer}",
    dissolve_field="agency",
    statistics_fields=None,
    multi_part="MULTI_PART",
    concatenation_separator="",
    out_lineage_table=None
)

In [ ]:
# NEW CELL

title = 'Total Acres Treated by Agency (Treatment Index Agency Field)'

query = f'''

    SELECT 
    	agency,         
    	SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM
        {schema}.{ti_stats_table}
    GROUP BY 
        agency
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
# NEW CELL

title = 'Total Acres Treated by Agency (Treatment Index Agency Field Dissolved)'

query = f'''

SELECT 
    	agency,         
    	ST_Area(ST_Transform(shape, 5070)) / 4046.8564224 AS acres
    FROM
       {schema}.{ti_agency_dissolve_layer}
    ORDER BY 
        acres desc;

'''

execute_and_append_to_csv(conn, query, output_file, title)

In [ ]:
# NEW CELL

title = 'Total Funding by Agency'

query = f'''

    SELECT
        agency as agency,
        sum(total_cost) as total_funding
    FROM
        staging.{ti_stats_table}
    GROUP BY agency
    ORDER BY total_funding desc;
    
'''

execute_and_append_to_csv(conn, query, output_file, title)